# Custom Transformers (`FunctionTransformer`)

### 🛠️ Ye Kya Hai aur Kyu Use Hota Hai? (The Problem & Solution)
* **Problem:** Scikit-Learn me `SimpleImputer` ya `StandardScaler` jaise in-built tools bohot achhe hain, lekin kabhi-kabhi hume data par kuch apna khud ka **custom mathematical logic** lagana hota hai. Agar hum directly Python function likh kar data badal denge, to wo function `Pipeline` ke andar fit nahi baithega (kyunki pipeline ko `.fit()` aur `.transform()` wale objects chahiye hote hain).
* **Solution:** **Custom Transformers** (jaise `FunctionTransformer`) kisi bhi normal Python function ko ek Scikit-Learn transformer me convert kar dete hain. Isse aap apne custom code ko seamlessly ML Pipeline ka hissa bana sakte hain.

---

### 💼 Kaha Use Hoga? (Real-World Use Cases)
ye 3 main situations me sabse zyada kaam aata hai:

1. **Heterogeneous Data (Mix Data):** Jab dataset me alag-alag tarah ka data ho (jaise ek column me images aur dusre me text). Aise me standard scalers fail ho jate hain, aur hume custom code likhna padta hai.
2. **Column-Specific Pipelines:** Jab aapka data ek Pandas DataFrame me ho aur aap chahte hain ki 'Age' column par alag math function lage aur 'Salary' par alag.
3. **Stateless Transformations:** Ye sabse important use case hai. "Stateless" ka matlab hai ki function ko data se kuch "seekhna" ya "yaad" nahi rakhna padta (jaise mean ya standard deviation). Wo bas data dekhta hai aur direct formula laga deta hai (Example: kisi column ka Logarithm nikalna).

---

### ⚙️ Wine Quality Dataset: Log Transformation Example
Agar aapka data bohot zyada faila hua hai (highly skewed), to ML models theek se kaam nahi karte. Isko theek karne ke liye hum data ka **Log** nikalte hain. `FunctionTransformer` ki madad se hum Numpy ke `np.log1p` (Log of 1+x) function ko transformer me badal sakte hain.

*(Yahan `np.log1p` isliye use karte hain taaki agar data me value `0` ho, to infinity error na aaye).*

---

In [7]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import FunctionTransformer

data = {
    'fixed_acidity': [7.4, 7.8, 7.8, 11.2],
    'volatile_acidity': [0.70, 0.88, 0.76, 0.28],
    'citric_acid': [0.00, 0.00, 0.04, 0.56],
    'residual_sugar': [1.9, 2.6, 2.3, 1.9],
    'chlorides': [0.076, 0.098, 0.092, 0.075],
    'free_sulfur_dioxide': [11.0, 25.0, 15.0, 17.0]
}

data = pd.DataFrame(data)

log_transformer = FunctionTransformer(np.log1p)
data_transformed = log_transformer.fit_transform(data)
data_transformed

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide
0,2.128232,0.530628,0.000000,1.064711,0.073250,2.484907
1,2.174752,0.631272,0.000000,1.280934,0.093490,3.258097
2,2.174752,0.565314,0.039221,1.193922,0.088011,2.772589
3,2.501436,0.246860,0.444686,1.064711,0.072321,2.890372


# Polynomial Features

### 🛠️ Ye Kya Hai aur Kyu Use Hota Hai? (The Problem & Solution)
* **Problem:** Zyadatar simple Machine Learning models (jaise Linear Regression) sirf straight-line (linear) relationships ko samajh pate hain. Lekin real-world data hamesha seedha nahi hota; usme curves aur complex patterns hote hain. Agar hum seedha model tede data par lagayenge, to accuracy bohot kharab aayegi (Underfitting).
* **Solution:** **Polynomial Features** hamare purane features ko aapas me multiply karke aur unki power (square, cube) nikal kar naye features generate kar deta hai. Isse hamara simple linear model bhi complex aur curved data ko aasani se samajh aur fit kar pata hai.

---

### 💼 Kaha Use Hoga? (Real-World Use Cases)
1. **Non-Linear Data modeling me:** Jab aapko pata ho ki ek feature ka asar doosre par linear nahi hai (e.g., speed aur braking distance ka relation square me badhta hai).
2. **Feature Interaction Pakadne me:** Jab do features mil kar kisi teesre output ko affect karte hain (jaise Ghar ki height aur width alag-alag itne important nahi hain, par unka multiplication yani 'Area' bohot important hai).

---

### ⚙️ Ye Kaam Kaise Karta Hai? (The Math Logic)
Maan lijiye aapke paas 2D data hai: `[a, b]` aur aapne `degree=2` set kiya hai.
Polynomial Features is data ko 6 naye hisso me tod dega:
`[1, a, b, a^2, ab, b^2]`

* `1`: Bias (intercept) ke liye.
* `a, b`: Original features.
* `a^2, b^2`: Features ke squares.
* `ab`: Dono features ka aapas me multiplication (Interaction term).

---


In [9]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures

# 1. Wine dataset load karna aur ';' se separate karna
wine_data = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv", sep=";")

# 2. Target variable ('quality') ko drop karna taaki sirf features bachein
wine_data = wine_data.drop(['quality'], axis=1)

print('Number of features before transformation =', wine_data.shape)

# 3. Polynomial Features object banana (degree 2 ke sath)
poly = PolynomialFeatures(degree=2)

# 4. Data par fit aur transform lagana
poly_wine_data = poly.fit_transform(wine_data)

print('Number of features after transformation =', poly_wine_data.shape)

# 5. Naye banaye gaye features ke naam dekhna
print("\nFirst 10 new feature names:")
print(poly.get_feature_names_out()[:10])

Number of features before transformation = (1599, 11)
Number of features after transformation = (1599, 78)

First 10 new feature names:
['1' 'fixed acidity' 'volatile acidity' 'citric acid' 'residual sugar'
 'chlorides' 'free sulfur dioxide' 'total sulfur dioxide' 'density' 'pH']


# Discretization (Binning / Quantization)

### 🛠️ Ye Kya Hai aur Kyu Use Hota Hai? (The Problem & Solution)
* **Problem:** Kabhi-kabhi continuous numbers (jaise Age ya Income) ka output ke sath sidha (linear) relation nahi hota. Jaise, ek 20 saal aur 25 saal ke insaan ki choice me utna farq nahi hoga, jitna 20 saal aur 60 saal ke insaan me hoga. Agar hum direct Linear Regression lagayenge, to model fail ho jayega.
* **Solution:** **Discretization (Binning)** continuous data ko chhote-chhote groups (buckets/bins) me tod deta hai. Ye continuous data ko *Categorical (Nominal)* data me badal deta hai.

---

### 🚀 Value Addition: Binning Ke Fayde (Why do it?)
1. **Introduces Non-Linearity:** Ye Linear models (jaise Logistic Regression) me ek "non-linear" dimag daal deta hai, jisse model alag-alag bins ke liye alag-alag weights (importance) seekh pata hai.
2. **Handles Outliers Automatically:** Jab data bins me chala jata hai, to bohot bade outliers (jaise 10 lakh salary) apne aakhri bin ka hissa ban jate hain aur model ko kharab nahi karte.

---

### 🧠 Value Addition: Binning Ki Strategies (Scikit-Learn Secret)
`KBinsDiscretizer` me data ko dabbo me baantne ke 3 tareeke hote hain (`strategy` parameter):
* **`uniform`:** Saare bins ki chaurayi (width) barabar hoti hai. (e.g., 0-10, 10-20, 20-30).
* **`quantile` (Default & Best):** Saare bins me number of data points barabar hote hain. (Outliers wale data ke liye sabse best).
* **`kmeans`:** Ye data me 1D KMeans clustering lagata hai aur similar values ko ek bin me daalta hai.

---



In [41]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer

# Dummy data create kar rahe hain (Taaki code direct chal sake)
wine_data = pd.DataFrame({
    'chlorides': [0.076, 0.098, 0.092, 0.075, 0.200, 0.050, 0.088]
})

# 1. Discretizer Object Banana
# n_bins=10 : Data ko 10 dabbo me baatna hai
# encode="onehot" : Output ko 0 aur 1 (matrix) me dega
# strategy="quantile" : Value Addition (Sabse balanced bins banayega)
enc = KBinsDiscretizer(n_bins=3, encode="onehot", strategy="quantile")

# 2. Scikit-Learn ko 2D data chahiye, isliye [['chlorides']] double bracket use kiya
X = wine_data[['chlorides']]

# 3. Fit aur Transform karna
X_binned = enc.fit_transform(X)

# SPARSE MATRIX LOGIC:
# Output ek 'Sparse Matrix' hota hai. Kyunki zyada values '0' hoti hain, 
# Numpy memory bachane ke liye unhe compress kar deta hai.
print("--- Compressed Sparse Matrix Format ---")
print(X_binned)

--- Compressed Sparse Matrix Format ---
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 7 stored elements and shape (7, 3)>
  Coords	Values
  (0, 1)	1.0
  (1, 2)	1.0
  (2, 2)	1.0
  (3, 0)	1.0
  (4, 2)	1.0
  (5, 0)	1.0
  (6, 1)	1.0


c:\Users\sumit\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(


# Handling Categorical Features 

### 🛠️ Ye Kya Hai aur Kyu Use Hota Hai? (The Problem & Solution)
* **Problem:** Machine Learning models ko sirf aur sirf **Numbers (Mathematics)** samajh aate hain. Lekin hamare real-world datasets me bohot saara data text ya categories me hota hai (jaise Education Level, City, Gender, ya State). Agar hum is text ko direct model me dalenge, to wo error dega.
* **Solution:** Hume in Text/Categorical features ko kisi tarah Numbers me convert karna hota hai. Iske 4 main tarike hote hain:
  1. Ordinal Encoding
  2. One-Hot Encoding
  3. Label Encoding
  4. Dummy Variables (Pandas `pd.get_dummies`)

---

#### Ordinal & OneHot Encoder

In [50]:
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder
import pandas as pd

url='https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data'

cols = ['sepal length', 'sepal width', 'petal length', 'petal width', 'label' ]
iris_data = pd.read_csv(url, header=None, names=cols)
iris_data.head()

,sepal length,sepal width,petal length,petal width,label
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [51]:

onehotencoder = OneHotEncoder(categories='auto')
print('Shape of y before encoding', iris_data. label. shape)

'''Passing 1d arrays as data to onehotencoder is deprecated in version ,
hence reshape to (-1,1) to have two dimensions.
Input of onehotencoder fit transform must not be 1-rank array'''

iris_labels = onehotencoder.fit_transform(iris_data.label.values.reshape(-1,1) )

# y.reshape(-1,1) is a 450x1 sparse matrix of type '<class 'numpy. float64'>'
# with 150 stored elements in Coordinate format.
# y is a 150x3 sparse matrix of type '<class 'numpy.float64'>' with 150 stored
# elements in compressed sparse row format.
print('Shape of y after encoding', iris_labels.shape)

# since output is sparse use to_array() to expand it.
print ("First 5 labels:")
print(iris_labels.toarray ( ) [ : 5] )


Shape of y before encoding (150,)
Shape of y after encoding (150, 3)
First 5 labels:
[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]]


In [52]:
enc = OrdinalEncoder()
iris_labels = np.array(iris_data[ 'label' ])

iris_labels_transformed = enc.fit_transform(iris_labels.reshape(-1, 1))
print ("Unique labels:", np.unique(iris_labels_transformed))

print ("\nFirst 5 labels:")
print (iris_labels_transformed [ : 5])

Unique labels: [0. 1. 2.]

First 5 labels:
[[0.]
 [0.]
 [0.]
 [0.]
 [0.]]


#### Label Encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

# get the class column in a new variable
iris_labels = np.array(iris_data['label' ])

# encode the class names to integers
enc = LabelEncoder()
label_integer = enc.fit_transform(iris_labels)

label_integer

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

#### MultiLabelBinerizer

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

movie_genres =[
    {'action', 'comedy' },
    {'comedy' } ,
    {'action', 'thriller'},
    {'science-fiction', 'action', 'thriller'}
]

mlb = MultiLabelBinarizer()

mlb.fit_transform(movie_genres)

array([[1, 1, 0, 0],
       [0, 1, 0, 0],
       [1, 0, 0, 1],
       [1, 0, 1, 1]])

## Composite Transformers

In [76]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MaxAbsScaler , OneHotEncoder



x = [
    [20.0, 'male',],
    [11.2, 'female',],
    [15.6, 'female',],
    [13.0, 'male', ],
    [18.6, 'male',],
    [16.4, 'female',]
]

X = np.array(x)

ct = ColumnTransformer([
    ('scaler' , MaxAbsScaler() , [0]),
    ('pass' , 'passthrough' , [0]),
    ('encoder' , OneHotEncoder() , [1])
])

transformed_X = ct.fit_transform(X)
pd.DataFrame(transformed_X)

,0,1,2,3
0,1.0,20.0,0.0,1.0
1,0.5599999999999999,11.2,1.0,0.0
2,0.78,15.6,1.0,0.0
3,0.65,13.0,0.0,1.0
4,0.93,18.6,0.0,1.0
5,0.82,16.4,1.0,0.0


#### Targeted Regressor

In [82]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X, y = X[:2000, : ], y[ :2000] # select a subset of data

transformer = MaxAbsScaler()

# Two regressor - one based on the original label.
regressor = LinearRegression()

# second regressor with transformed labels.
regr = TransformedTargetRegressor (regressor=regressor, transformer=transformer)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
regr.fit(X_train, y_train)
print('R2 score of raw label regression: {0:.2f}'.format(regr.score(X_test, y_test)))

raw_target_regr = LinearRegression() .fit(X_train, y_train)
print('R2 score of transformed label regression: {0:.2f}'.format(raw_target_regr.score(X_test, y_test)))

R2 score of raw label regression: 0.59
R2 score of transformed label regression: 0.59
